# Step 2: Run the Zerobus ingest

This is the **single notebook** that runs the Zerobus ingest for the demo. The SEPA Delta table is created in **01_train_model_and_pipeline**; this notebook uses the **same catalog and schema** so both the pipeline and ingest target the same table (`catalog.schema.sepa_transactions`).

**Once ingest is running:** The Delta Live Tables pipeline (managed by `databricks.yml`) reads from this table in continuous mode. You will see transactions flowing: bronze -> silver (with ML risk scoring) -> gold. Open the **Pipelines** UI to watch progress.

**Prerequisites:**
- Zerobus Ingest enabled (Public Preview).
- Service principal with MODIFY and SELECT on the target table.
- **Run in Databricks Workspace** (no Databricks Connect).

In [ ]:
%pip install 'databricks-sdk[notebook]' 'databricks-zerobus-ingest-sdk'

## Configuration

**Workspace URL** and **Server endpoint** are read from the workspace when this notebook runs in Databricks (no widgets needed).

**Credentials** use a **service principal**. Create one in Databricks, then enter its **Client ID** and **Client secret** in the widgets (or store them in a secret scope). See the **Credentials** section below.

| Widget | How to obtain |
|--------|----------------|
| **Secret scope** | Optional. If set, **Client ID** and **Client secret** are read from secrets `zerobus_client_id` and `zerobus_client_secret` when their widgets are empty. |
| **Client ID** | Service principal application ID (see below). |
| **Client secret** | Service principal secret (see below). |
| **Zerobus region** | Optional. Override when auto-detected region is wrong (e.g. `eu-west-1`, `ap-southeast-1`). Leave empty to use URL or Spark config. |

### How to get Client ID and Client secret

See the **Credentials** section below for how to create a service principal and get Client ID and secret.


Load shared demo settings (catalog, schema, table names) and SEPA transaction constants (companies, locations, currencies).

In [ ]:
%run ../_shared/config

In [ ]:
table_name = SEPA_TABLE_FULL

# Create widgets for credentials (workspace URL and server endpoint are read from the workspace).
dbutils.widgets.text("zerobus_secret_scope", "", "1. Secret scope for credentials (optional; keys: zerobus_client_id, zerobus_client_secret)")
dbutils.widgets.text("client_id", "", "2. Client ID (or from secret scope above; also used for grants)")
dbutils.widgets.text("client_secret", "", "3. Client secret (or from secret scope above)")
dbutils.widgets.text("zerobus_region", "", "4. Zerobus region override (e.g. eu-west-1, ap-southeast-1); leave empty to auto-detect from workspace URL")

In [ ]:
%run ../_shared/sepa_constants

### Workspace and Zerobus endpoint

The next cell reads your **workspace URL** and **Zerobus Ingest server endpoint** from the Databricks workspace. No configuration is required: the notebook uses the workspace it runs in and builds the Zerobus endpoint from the workspace ID and region (e.g. `https://<workspace-id>.zerobus.<region>.cloud.databricks.com`). Run this cell in Databricks; it will fail if run outside the workspace.

In [ ]:
# Workspace URL and Zerobus endpoint (from the current Databricks workspace).
from dbruntime.databricks_repl_context import get_context
from databricks.sdk import WorkspaceClient
import re

w = WorkspaceClient()
host = spark.conf.get("spark.databricks.workspaceUrl", "").strip()
if not host:
    raise RuntimeError("spark.databricks.workspaceUrl not set. Run this notebook in Databricks Workspace.")
workspace_url = host if host.startswith("http") else f"https://{host}"
workspace_id = get_context().workspaceId

# Optional override from widget
region_override = (dbutils.widgets.get("zerobus_region") or "").strip()

is_azure = "azuredatabricks.net" in host
dns_suffix = "azuredatabricks.net" if is_azure else "cloud.databricks.com"

# AWS region pattern (e.g. us-west-2, eu-central-1, ap-southeast-1)
_AWS_REGION_RE = re.compile(r"^[a-z]{2}-[a-z]+-\d+$")

def _detect_region():
    if region_override:
        return region_override
    # AWS: try to extract region from hostname (e.g. dbc-xxx.us-west-2.cloud.databricks.com)
    if not is_azure:
        parts = host.replace(f".{dns_suffix}", "").split(".")
        if len(parts) >= 2 and _AWS_REGION_RE.match(parts[-1]):
            return parts[-1]
    # Fallback for both clouds: get region from Unity Catalog metastore summary (no admin required)
    try:
        summary = w.api_client.do("GET", "/api/2.1/unity-catalog/metastore_summary")
        region = summary.get("region", "").strip()
        if region:
            return region
    except Exception:
        pass
    return None

region = _detect_region()
if not region:
    raise ValueError(
        "Could not auto-detect region. Please enter your region in the 'Zerobus region' widget "
        "(e.g. westeurope, eastus2, us-west-2). "
        "See: https://docs.databricks.com/en/ingestion/zerobus-overview"
    )

server_endpoint = f"https://{workspace_id}.zerobus.{region}.{dns_suffix}"

print("Workspace URL:  ", workspace_url)
print("Server endpoint:", server_endpoint)
print("Region:         ", region)

### Credentials

Use a **service principal** for Zerobus Ingest. Fill in **Client ID** and **Client secret** in the widgets below (or set **Secret scope** and store them there). Re-run the widget cell above if you change the values.

### How to get Client ID and Client secret

1. **Create a service principal** in Databricks: **Settings -> Identity and access -> Service principals -> Add service principal** (or [Service Principals API](https://docs.databricks.com/api/workspace/serviceprincipals/create)).
2. **Add the SP to this workspace.** If you created the SP at the account level, also add it to the workspace where this notebook runs (**Settings -> Identity and access -> Service principals -> Add service principal -> Add existing**). If the SP isn't in the workspace, grants you issue below resolve to no one, and Zerobus's OAuth token request fails with `invalid_authorization_details` (401).
3. **Generate an OAuth secret** for it: open the service principal -> **Secrets** tab -> **Generate secret**. Copy the **Client ID** (application ID) and **secret** — the secret is shown only once.
4. The grant cell below issues `USE CATALOG`, `USE SCHEMA`, and `SELECT, MODIFY` on the target table. All three are required for Zerobus's UC token request to succeed.

Enter **Client ID** and **Client secret** in the widgets below, or store them in a [secret scope](https://docs.databricks.com/en/security/secrets/secret-scopes.html) with keys `zerobus_client_id` and `zerobus_client_secret` and set **Secret scope**.

**Docs:** [Create service principal & OAuth secret](https://docs.databricks.com/en/dev-tools/auth/oauth-m2m.html) · [Zerobus Ingest](https://docs.databricks.com/en/ingestion/zerobus-ingest)

In [ ]:
# Credentials: from widgets or from secret scope.
zerobus_secret_scope = dbutils.widgets.get("zerobus_secret_scope").strip()
client_id = dbutils.widgets.get("client_id").strip()
client_secret = dbutils.widgets.get("client_secret").strip()

if (not client_id or not client_secret) and zerobus_secret_scope:
    try:
        if not client_id:
            client_id = dbutils.secrets.get(zerobus_secret_scope, "zerobus_client_id")
        if not client_secret:
            client_secret = dbutils.secrets.get(zerobus_secret_scope, "zerobus_client_secret")
        if client_id and client_secret:
            print("Credentials loaded from secret scope:", zerobus_secret_scope)
    except Exception as e:
        if not client_id or not client_secret:
            raise ValueError(f"Secret scope '{zerobus_secret_scope}' missing or keys zerobus_client_id/zerobus_client_secret not found: {e}") from e

if not client_id or not client_secret:
    raise ValueError(
        "Fill in Client ID and Client secret (widgets or Secret scope). See the Configuration section above."
    )

### Grant access to service principal (Client ID)

After creating a service principal, run the cell below to grant its **Client ID** (application ID) access to the catalog, schema, and SEPA table so it can use Zerobus Ingest. Use the same **Client ID** as in the widget above; run the grant cell after entering it.

In [ ]:
# Grant USE CATALOG, USE SCHEMA, and SELECT/MODIFY on the table to the service principal
# by Client ID. All three are required for Zerobus's UC token request to succeed; missing
# USE CATALOG is the most common cause of `invalid_authorization_details` (401) from Zerobus.
client_id_grant = (dbutils.widgets.get("client_id") or "").strip() or globals().get("client_id", "")
if not client_id_grant:
    print("No Client ID set. Set the 'Client ID' widget (same as for Zerobus credentials) and re-run to grant access.")
else:
    # Escape backticks in the client ID for SQL
    client_id_sql = client_id_grant.replace("`", "``")
    spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{client_id_sql}`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{client_id_sql}`")
    spark.sql(f"GRANT SELECT, MODIFY ON TABLE {SEPA_TABLE_FULL} TO `{client_id_sql}`")
    print(f"Granted USE CATALOG, USE SCHEMA, and SELECT/MODIFY on table to Client ID: {client_id_grant}")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}");

In [ ]:
# Table is created in 01_setup_model_and_pipeline. Ensure it exists (e.g. if running this notebook alone).
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_name} (
    settlement_amount DOUBLE,
    currency STRING,
    debtor STRING,
    debtor_location STRING,
    creditor STRING,
    creditor_location STRING,
    intermediary_agent STRING,
    remittance_information STRING,
    instruction_id STRING,
    ingestion_time TIMESTAMP
  )
""")
print("Table", table_name, "ready (created if not existed).")

In [ ]:
# COMPANIES, DEBTOR_LOCATIONS, CREDITOR_LOCATIONS, INTERMEDIARY_AGENTS,
# REMITTANCE_TYPES, CURRENCIES are loaded from sepa_constants (%run above).
import random
import string

NUM_RECORDS = 20  # number of SEPA events to ingest

## Run Zerobus ingest

Create a Zerobus stream to the SEPA transactions table and ingest synthetic SEPA records. The code pushes `NUM_RECORDS` random transactions into the table; the DLT pipeline (managed by `databricks.yml`) will read from this table and process them. Run the cell below to start ingesting.

**If you see `ZerobusException: ... tls handshake eof`:** Run this notebook on a **Databricks cluster** (not locally and not with Databricks Connect). The Zerobus endpoint is only reachable from the Databricks environment. Also ensure Zerobus Ingest is enabled for your workspace; if the error persists, check firewall/VPN or contact your admin.

In [ ]:
# Stream targets the table in table_name (main.kyc_demo.sepa_transactions). Run "Create table" first.
import time
from datetime import datetime, timezone
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

# Close any existing stream from a previous run so the new stream sees the current table schema.
try:
    if "stream" in globals() and stream is not None:
        stream.close()
except Exception:
    pass
time.sleep(2)  # allow server to release previous stream/schema

sdk = ZerobusSdk(server_endpoint, workspace_url)
table_properties = TableProperties(table_name)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(client_id, client_secret, table_properties, options)

try:
    for i in range(NUM_RECORDS):
        debtor = random.choice(COMPANIES)
        creditor = random.choice([c for c in COMPANIES if c != debtor] or COMPANIES)
        currency = random.choice(CURRENCIES)
        amount = round(random.uniform(100.0, 100_000.0), 2)
        record = {
            "settlement_amount": amount,
            "currency": currency,
            "debtor": debtor,
            "debtor_location": random.choice(DEBTOR_LOCATIONS),
            "creditor": creditor,
            "creditor_location": random.choice(CREDITOR_LOCATIONS),
            "intermediary_agent": random.choice(INTERMEDIARY_AGENTS),
            "remittance_information": f"{random.choice(REMITTANCE_TYPES)}",
            "instruction_id": "".join(random.choices(string.ascii_uppercase + string.digits, k=9)),
            "ingestion_time": int(datetime.now(timezone.utc).timestamp() * 1_000_000),  # epoch microseconds for TIMESTAMP
        }
        ack = stream.ingest_record(record)
        ack.wait_for_ack()
        time.sleep(random.uniform(0.5, 1.5))  # ~1 row per second on average, randomized
finally:
    stream.close()

## Disable pipeline

Stop the DLT pipeline so it is not left running after the demo. Run this cell when you are done with the Zerobus ingest.

In [ ]:
# Stop the SEPA KYC Demo Pipeline (matches dev prefix "[dev <username>] ..." too)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
pipeline_id = None
for p in w.pipelines.list_pipelines():
    if p.name and PIPELINE_NAME in p.name:
        pipeline_id = p.pipeline_id
        break
if pipeline_id:
    w.pipelines.stop(pipeline_id)
    print("Pipeline stopped:", p.name)
else:
    print("Pipeline not found:", PIPELINE_NAME, "(may already be stopped or not created yet)")

## Cleanup

Drop all demo resources: the DLT pipeline, the MLflow model, the schema (and all tables in it), and the MLflow experiment. Run this cell to tear down everything created by the demo.

In [ ]:
from databricks.sdk import WorkspaceClient
from urllib.parse import quote

w = WorkspaceClient()
_current_user = w.current_user.me().user_name or "unknown"
errors = []

# 1. Delete the DLT pipeline (matches dev prefix "[dev <username>] ..." too)
try:
    for p in w.pipelines.list_pipelines():
        if p.name and PIPELINE_NAME in p.name:
            w.pipelines.delete(p.pipeline_id)
            print(f"Deleted pipeline: {p.name}")
            break
    else:
        print(f"Pipeline not found: {PIPELINE_NAME}")
except Exception as e:
    errors.append(f"Pipeline delete: {e}")

# 2. Delete the registered model from Unity Catalog (delete all versions first)
model_name = f"{CATALOG}.{SCHEMA}.sepa_fraud_model"
model_name_encoded = quote(model_name, safe="")
try:
    versions = w.api_client.do("GET", f"/api/2.1/unity-catalog/models/{model_name_encoded}/versions")
    for v in versions.get("model_versions", []):
        ver = v.get("version")
        w.api_client.do("DELETE", f"/api/2.1/unity-catalog/models/{model_name_encoded}/versions/{ver}")
    w.api_client.do("DELETE", f"/api/2.1/unity-catalog/models/{model_name_encoded}")
    print(f"Deleted model: {model_name}")
except Exception as e:
    errors.append(f"Model delete: {e}")

# 3. Delete the MLflow experiment (via REST API)
try:
    exp_resp = w.api_client.do("GET", "/api/2.0/mlflow/experiments/get-by-name", query={"experiment_name": f"/Users/{_current_user}/kyc_aml_sepa_fraud_demo"})
    exp_id = exp_resp.get("experiment", {}).get("experiment_id")
    if exp_id:
        w.api_client.do("POST", "/api/2.0/mlflow/experiments/delete", body={"experiment_id": exp_id})
        print(f"Deleted experiment: /Users/{_current_user}/kyc_aml_sepa_fraud_demo")
    else:
        print("Experiment not found")
except Exception as e:
    errors.append(f"Experiment delete: {e}")

# 4. Drop the schema and all tables within it
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
    print(f"Dropped schema: {CATALOG}.{SCHEMA}")
except Exception as e:
    errors.append(f"Schema drop: {e}")

if errors:
    print("\nSome cleanup steps failed:")
    for err in errors:
        print(f"  - {err}")
else:
    print("\nAll demo resources cleaned up.")